# Practical Problems 5 — Alignment & Phylogeny
## TL;DR — Plain English Introduction

**What is sequence alignment?** Finding the best way to match two sequences by allowing insertions, deletions (gaps), and mismatches. "ATCG" vs "AGCG" → align them to see they differ at position 2.

**Three alignment types:**
- **Global** (Needleman-Wunsch): align entire sequences end-to-end
- **Local** (Smith-Waterman): find the best matching region
- **Affine gap** (Gotoh): penalize opening a gap more than extending it (biologically realistic)

**No Python background?**
```python
# Dynamic programming table for alignment
# dp[i][j] = best score to align seq1[:i] with seq2[:j]
dp = [[0]*(len(seq2)+1) for _ in range(len(seq1)+1)]
# Fill left column and top row (gap penalties)
# Then fill the table using the recurrence
```

**What is phylogeny?** Building the evolutionary tree of species/proteins from their sequences. UPGMA = simple hierarchical clustering on distance matrix.

**What you'll solve:**
- HackerRank: edit distance, LCS ("Common Child")
- Rosalind: GLOB, LOCA, GCON, FITT alignments; UPGMA, neighbor-joining trees; TRIE, KMP

**Learning path:** 01/01 (Alignment theory) → This notebook → 01/05 (Phylogeny advanced) → 07/03 (AF3 evaluation)

## Beginner Teaching Frame

**Who should fully work through this notebook:** every student. This is where fluency is built.

**How to study it on a first pass:**
- try each problem before reading the answer
- keep a timer for at least one section
- write the complexity and the biological interpretation after each solution

**Common traps:** reading solution code too early, confusing recognition with mastery, and skipping timed practice because the answers are available.


## Canonical Resource Upgrade

Use these as practice companions:
- [HackerRank](https://www.hackerrank.com/domains)
- [Rosalind](https://rosalind.info/problems/list-view/)
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/)


# Practical 5 — Sequence Alignment & Phylogeny

## What This Notebook Is

All alignment variants and phylogenetic methods in one place. HackerRank tests the algorithms; Rosalind provides the biological problems. Together they cover every alignment scenario you'll encounter in an interview.

**Tags**: `[HR]` = HackerRank | `[R:ID]` = Rosalind problem ID

## Learning Objectives
- [ ] Implement all 6 alignment variants (global, local, fitting, overlap, semiglobal, affine-gap)
- [ ] Build suffix arrays and apply BWT for text compression
- [ ] Compute evolutionary distances with Jukes-Cantor correction
- [ ] Build phylogenetic trees from distance matrices
- [ ] Compute small parsimony on given trees

## The Alignment Decision Tree
```
Compare two FULL sequences? → Global (Needleman-Wunsch)
Find best REGION of similarity? → Local (Smith-Waterman)
Fit a SHORT read into a GENOME? → Fitting alignment
Assemble OVERLAPPING reads? → Overlap alignment
Align sequences of DIFFERENT lengths? → Semiglobal
Biological gaps (open + extend)? → Affine gap penalty
```

## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 01/01 (sequence biology), 01/04 (pairwise alignment intro), 01/05 (multiple alignment intro) |
| 📍 You Are Here | Module 08/05 — Alignment & Phylogeny |
| ➡️ Next Steps | 09/01 (model diagnostics — shift from bioinformatics algorithms to ML) |
| 🏁 Goal | Implement all 5 alignment modes, KMP pattern matching, and UPGMA/NJ phylogenetic trees |

### 🆕 From Scratch? Start Here:
1. [Rosalind GLOB problem](https://rosalind.info/problems/glob/) — global alignment as entry point
2. [Ben Langmead alignment lectures](https://www.youtube.com/playlist?list=PL2mpR0RYFQsBiCWVJSvVAO3OJ2t7DzoHA) — YouTube, Johns Hopkins
3. 01/04 (this repo) — pairwise alignment fundamentals
4. This notebook — all alignment modes + phylogeny
**Time:** 12–18h | **Difficulty:** ⭐⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 01/01 (sequence biology), 01/04 (pairwise alignment), 01/05 (multiple alignment), 08/01 (string algorithms: KMP)
- Used by: 07/02 (AF3 uses sequence alignment for MSA construction), 07/03 (structural alignment metrics parallel sequence alignment)
- Parallel: 01/04 (pairwise alignment — this notebook covers all modes including those in 01/04)

In [ ]:
# Alignment Section 1 — Global + Local alignment
import numpy as np

def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m+1, n+1))
    for i in range(m+1): dp[i][0] = i * gap
    for j in range(n+1): dp[0][j] = j * gap
    for i in range(1, m+1):
        for j in range(1, n+1):
            diag = dp[i-1][j-1] + (match if seq1[i-1]==seq2[j-1] else mismatch)
            dp[i][j] = max(diag, dp[i-1][j]+gap, dp[i][j-1]+gap)
    # Traceback
    aligned1, aligned2 = '', ''
    i, j = m, n
    while i > 0 or j > 0:
        if i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+(match if seq1[i-1]==seq2[j-1] else mismatch):
            aligned1 = seq1[i-1]+aligned1; aligned2 = seq2[j-1]+aligned2; i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+gap:
            aligned1 = seq1[i-1]+aligned1; aligned2 = '-'+aligned2; i-=1
        else:
            aligned1 = '-'+aligned1; aligned2 = seq2[j-1]+aligned2; j-=1
    return dp[m][n], aligned1, aligned2

score, a1, a2 = needleman_wunsch("PLEASANTLY","MEANLY")
print(f"Global alignment score: {score}")
print(f"  {a1}")
print(f"  {a2}")

In [ ]:
# Alignment Section 2 — Smith-Waterman (local alignment)
import numpy as np

def smith_waterman(seq1, seq2, match=2, mismatch=-1, gap=-2):
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m+1, n+1))
    best_score, best_pos = 0, (0, 0)
    for i in range(1, m+1):
        for j in range(1, n+1):
            diag = dp[i-1][j-1] + (match if seq1[i-1]==seq2[j-1] else mismatch)
            dp[i][j] = max(0, diag, dp[i-1][j]+gap, dp[i][j-1]+gap)
            if dp[i][j] > best_score:
                best_score = dp[i][j]; best_pos = (i, j)
    # Traceback from best position
    aligned1, aligned2 = '', ''
    i, j = best_pos
    while dp[i][j] > 0:
        if i>0 and j>0 and dp[i][j]==dp[i-1][j-1]+(match if seq1[i-1]==seq2[j-1] else mismatch):
            aligned1 = seq1[i-1]+aligned1; aligned2 = seq2[j-1]+aligned2; i-=1; j-=1
        elif i>0 and dp[i][j]==dp[i-1][j]+gap:
            aligned1 = seq1[i-1]+aligned1; aligned2 = '-'+aligned2; i-=1
        else:
            aligned1 = '-'+aligned1; aligned2 = seq2[j-1]+aligned2; j-=1
    return best_score, aligned1, aligned2

score, a1, a2 = smith_waterman("ATAGCTGCATG","CGCATGG")
print(f"Local alignment score: {score}")
print(f"  {a1}")
print(f"  {a2}")

In [ ]:
# Alignment Section 3 — Affine gap alignment (Rosalind GCON/GAFF)
import numpy as np

def affine_gap_align(seq1, seq2, match=1, mismatch=-1, gap_open=-11, gap_extend=-1):
    """Gotoh algorithm for global alignment with affine gap penalty."""
    m, n = len(seq1), len(seq2)
    NEG_INF = float('-inf')
    M = np.full((m+1,n+1), NEG_INF)   # match/mismatch state
    X = np.full((m+1,n+1), NEG_INF)   # gap in seq2 (deletion in seq1)
    Y = np.full((m+1,n+1), NEG_INF)   # gap in seq1 (insertion)
    M[0][0] = 0

    for i in range(1, m+1):
        X[i][0] = gap_open + gap_extend * i
    for j in range(1, n+1):
        Y[0][j] = gap_open + gap_extend * j

    for i in range(1, m+1):
        for j in range(1, n+1):
            s = match if seq1[i-1]==seq2[j-1] else mismatch
            M[i][j] = s + max(M[i-1][j-1], X[i-1][j-1], Y[i-1][j-1])
            X[i][j] = max(M[i-1][j] + gap_open + gap_extend,
                          X[i-1][j] + gap_extend)
            Y[i][j] = max(M[i][j-1] + gap_open + gap_extend,
                          Y[i][j-1] + gap_extend)

    best = max(M[m][n], X[m][n], Y[m][n])
    return best

score = affine_gap_align("PLEASANTLY","MEANLY")
print(f"Affine gap alignment score: {score}")
score2 = affine_gap_align("ACGT","AGT")
print(f"Affine gap 'ACGT' vs 'AGT': {score2}")

In [ ]:
# Alignment Section 4 — Trie construction + Phylogenetic distance
# --- Rosalind TRIE ---
def build_trie(patterns):
    trie = {0: {}}
    node_count = 0
    for pattern in patterns:
        node = 0
        for char in pattern:
            if char not in trie[node]:
                node_count += 1
                trie[node][char] = node_count
                trie[node_count] = {}
            node = trie[node][char]
    return trie

patterns = ["ATAGA","ATC","GAT"]
trie = build_trie(patterns)
print("Trie edges:")
for node, edges in sorted(trie.items()):
    for char, child in edges.items():
        print(f"  {node} --{char}--> {child}")

# --- JC + K2P distances (Rosalind TRAN, PDST) ---
import numpy as np
def jc_distance(s1, s2):
    p = sum(c1!=c2 for c1,c2 in zip(s1,s2)) / len(s1)
    return -3/4 * np.log(1 - 4*p/3) if p < 0.75 else float('inf')

def k2p_distance(s1, s2):
    ts = sum(1 for c1,c2 in zip(s1,s2) if {c1,c2} in [{'A','G'},{'C','T'}])
    tv = sum(1 for c1,c2 in zip(s1,s2) if c1!=c2 and {c1,c2} not in [{'A','G'},{'C','T'}])
    L = len(s1)
    P, Q = ts/L, tv/L
    try:
        return -0.5*np.log(1-2*P-Q) - 0.25*np.log(1-2*Q)
    except:
        return float('inf')

s1, s2 = "ACGTACGT", "ACGGACGT"
print(f"\nJC distance: {jc_distance(s1,s2):.4f}")
print(f"K2P distance: {k2p_distance(s1,s2):.4f}")

In [ ]:
# Alignment Section 5 — Neighbor-Joining + Error correction
import numpy as np

def neighbor_joining(D, labels):
    """Neighbor-Joining algorithm (Saitou & Nei 1987) — returns tree edges."""
    n = len(labels)
    D = [list(row) for row in D]
    active = list(range(n))
    edges = []
    node_id = n

    while len(active) > 2:
        m = len(active)
        r = [sum(D[active[i]][active[j]] for j in range(m)) / (m-2) for i in range(m)]
        # Q matrix
        best_q, best_i, best_j = float('inf'), 0, 1
        for i in range(m):
            for j in range(i+1, m):
                q = D[active[i]][active[j]] - r[i] - r[j]
                if q < best_q:
                    best_q, best_i, best_j = q, i, j
        i_idx, j_idx = active[best_i], active[best_j]
        d_ij = D[i_idx][j_idx]
        branch_i = d_ij/2 + (r[best_i]-r[best_j])/2
        branch_j = d_ij - branch_i
        edges.append((labels[i_idx] if i_idx < n else f'N{i_idx}', f'N{node_id}', round(branch_i,3)))
        edges.append((labels[j_idx] if j_idx < n else f'N{j_idx}', f'N{node_id}', round(branch_j,3)))

        new_row = [(D[i_idx][active[k]] + D[j_idx][active[k]] - d_ij)/2
                   for k in range(len(active)) if active[k] not in (i_idx, j_idx)]
        active = [a for a in active if a not in (i_idx, j_idx)] + [node_id]
        for row in D: row.append(0)
        D.append([0]*len(D[0]))
        for k, a in enumerate([x for x in range(len(active)-1)]):
            D[node_id][active[k]] = D[active[k]][node_id] = new_row[k]
        node_id += 1

    if len(active) == 2:
        i, j = active
        edges.append((labels[i] if i<n else f'N{i}', labels[j] if j<n else f'N{j}',
                      round(D[i][j],3)))
    return edges

labels = ['A','B','C','D']
D = [[0,5,9,9],[5,0,10,10],[9,10,0,8],[9,10,8,0]]
edges = neighbor_joining(D, labels)
print("NJ tree edges:")
for e in edges:
    print(f"  {e[0]} -- {e[2]} -- {e[1]}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Alignment modes**: global (Needleman-Wunsch), local (Smith-Waterman), semi-global (no end-gap penalty), fitting (query in reference), overlap
- **Profile HMMs**: match/insert/delete states, Viterbi (best path) vs Forward (sum over all paths) — foundation of HMMER
- **Jukes-Cantor model**: d = -3/4 ln(1 - 4p/3) — simplest nucleotide substitution model; basis for phylogenetic distance
- **UPGMA**: ultrametric tree assumption (constant rate), bottom-up merging by average pairwise distance — O(N³)
- **KMP failure function**: `fail[j]` = length of longest proper prefix of pattern[0..j] that is also a suffix — O(m) construction

### 2️⃣ Must-Have Popular Resources
- 📘 Biological Sequence Analysis (Durbin, Eddy, Krogh, Mitchison) — the definitive text; HMMs, DP, phylogenetics
- 📘 Bioinformatics Algorithms (Compeau & Pevzner) — free online; chapters on alignment, trees, pattern matching
- 🎓 Ben Langmead alignment lectures (YouTube) — Johns Hopkins; covers BWT, suffix arrays, bowtie2 internals
- ⭐ GitHub [biopython/biopython alignment](https://github.com/biopython/biopython) — PairwiseAligner with all modes
- ⭐ GitHub [etetoolkit/ete](https://github.com/etetoolkit/ete) — phylogenetic tree visualization and analysis

### 3️⃣ Practicals — Hands-On
- 💻 Rosalind alignment: GLOB (global), LOCA (local), GCON (global affine), FITT (fitting), OLAP (overlap) — all 5 modes
- 💻 Rosalind pattern matching: TRIE (trie construction), KMP (failure function), CORR (error correction)
- 💻 Rosalind phylogeny: PDST (pairwise distance), TRAN (transitions/transversions), UPGMA, NWCK (Newick format)
- 💻 HackerRank: Edit Distance (LCS variant), Common Child, Special String Again
- 📊 Kaggle: [Sequence alignment benchmarks](https://www.kaggle.com/search?q=sequence+alignment)

### 4️⃣ Real-World Problems
- 🏭 Clinical genomics: variant classification pipelines use Smith-Waterman for indel alignment (GATK IndelRealigner)
- 🏭 SARS-CoV-2 phylogeny: UShER uses parsimony-based tree placement to track 15M+ SARS-CoV-2 sequences in real time
- 🏭 Antibody humanization: alignment of CDR loops to human germline sequences using affine gap penalties
- 📊 Dataset: [SARS-CoV-2 GenBank sequences](https://www.ncbi.nlm.nih.gov/sars-cov-2/) — 10M+ sequences for phylogenetic analysis
- 🔬 Application: Comparative genomics — synteny blocks identified by all-vs-all protein alignment (OrthoFinder)

### 5️⃣ Interview Tips
- ❓ Must know: Global vs local vs semi-global choice — global for whole-sequence comparison (homologs), local for domain finding, semi-global for primer alignment
- ❓ Must know: KMP O(n+m) vs naive O(nm) — KMP never moves backwards in text; show the failure function for AABABC
- ❓ Must know: UPGMA vs Neighbor-Joining — UPGMA assumes molecular clock (constant rate); NJ does not, is more accurate for unequal rates
- ⚠️ Common mistake: Using global alignment for query vs database search — local (SW) is always correct for database search; global forces penalizing flanking sequence
- 💡 Pro tip: For affine gap penalties, you need three DP matrices (M, X, Y) — be able to sketch the recurrence in an interview

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [MMseqs2](https://github.com/soedinglab/MMseqs2) — ultra-fast protein/DNA sequence search (1000x faster than BLAST)
- 🔥 Trending: [DIAMOND](https://github.com/bbuchfink/diamond) — protein alignment 100x faster than BLAST, used for large metagenomes
- 🔥 Trending: [IQ-TREE 2](https://github.com/iqtree/iqtree2) — maximum likelihood phylogenetics; current standard for publication-quality trees
- 🚀 Build this: A pairwise aligner that supports all 5 modes (global/local/semi-global/fitting/overlap) using a single DP function with mode-controlled initialization and termination

In [ ]:
# Resources for 08/05
print("KEY RESOURCES — Module 08/05: Alignment + Phylogeny")
print()
refs = [
    "Rosalind: GLOB, LOCA, GCON, GAFF, PDST, TRAN, TRIE, NJ, UPGMA, CORR",
    "HackerRank: Common Child (LCS), Edit Distance",
    "EMBOSS: online NW/SW alignment tools",
    "Biopython pairwise2: in-memory alignment",
    "ETE3 / DendroPy: phylogenetic tree libraries",
    "MEGA: phylogenetics software suite",
    "IQ-TREE: maximum likelihood phylogenetics",
    "Rosalind Bioinformatics Algorithms book: bioinformaticsalgorithms.com",
]
for r in refs:
    print(f"  {r}")

print()
print("COMPLEXITY:")
algos = {
    "Needleman-Wunsch": "O(nm) time and space",
    "Smith-Waterman":   "O(nm) time and space",
    "Gotoh affine gap": "O(nm) time, 3 matrices",
    "KMP":             "O(n+m) time, O(m) space",
    "Trie":            "O(n*m) build, O(m) search per query",
    "UPGMA":           "O(n^2) per iteration = O(n^3) total",
    "NJ":              "O(n^3)",
}
for alg, c in algos.items():
    print(f"  {alg}: {c}")

## Interview Cheat Sheet — Alignment

| Mode | When to use | Init first row | Init first col | Score start |
|------|------------|----------------|----------------|-------------|
| Global (NW) | Compare full sequences | i × gap | j × gap | dp[m][n] |
| Local (SW) | Find best region | 0 | 0 | max of matrix |
| Fitting | Short read in genome | 0 | i × gap | max(dp[m,:]) |
| Overlap | Assembly | 0 | 0 | max last row/col |
| Semiglobal | Different lengths | i × gap | 0 | max last row |
| Affine gap | Biologically realistic | — | — | 3 DP tables |

**Time complexity**: All DP alignments are O(mn). Space: O(mn) for traceback, O(min(m,n)) if just score needed.

**Affine gap key insight**: 3 states (M=match, X=gap in s2, Y=gap in s1). Transition M→X costs gap_open; X→X costs gap_extend. This models biology where long gaps are rare but single-base gaps are costly to start.

## Real World Problems

### Problem 1: SARS-CoV-2 Variant Alignment
Download spike protein sequences of Alpha, Beta, Delta, Omicron variants. Perform global alignment (BLOSUM62) between each pair. Build a phylogenetic tree. Do the evolutionary distances match the known variant emergence order?

**Resources**: [NCBI Virus (SARS-CoV-2)](https://www.ncbi.nlm.nih.gov/labs/virus/) | [BioPython MSA tools (GitHub)](https://github.com/biopython/biopython) | [COVID variant dataset (Kaggle)](https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge)

### Problem 2: Multiple Sequence Alignment Benchmark
Download the BAliBASE benchmark (gold-standard MSA dataset). Implement progressive multiple alignment (build guide tree → align along tree). Measure sum-of-pairs score against the BAliBASE reference alignment.

**Resources**: [BAliBASE dataset](http://www.lbgi.fr/balibase/) | [ClustalW algorithm (GitHub)](https://github.com/GSLBiotech/clustal-omega) | [MSA benchmark (HuggingFace)](https://huggingface.co/datasets/tattabio/tRNA_msas)

### Problem 3: Rosalind Alignment Track
EDIT → EDTA → GLOB → LOCA → GCON → FITT → OLIN → OSYM → MGAP → IBAL → MULT

**Resources**: [rosalind.info alignment problems](https://rosalind.info/problems/list-view/)

## Mastery Check

Before moving on, make sure you can:
1. solve representative problems without looking up the answer
2. state the time complexity correctly
3. explain why your solution is correct
4. identify which earlier notebook you should revisit if a problem still feels opaque


---
## 🐛 Debug Exercise — Alignment and Phylogeny Bugs

Find and fix **3 bugs** in this Smith-Waterman and neighbor-joining implementation.

<details><summary>Show bugs</summary>

**Bug 1:** Smith-Waterman allows negative values in the DP matrix — local alignment must clamp to 0: `dp[i][j] = max(0, diag, up, left)`.

**Bug 2:** The traceback starts from the maximum value in the matrix, but the code starts from `dp[n][m]` (bottom-right corner) — that's Needleman-Wunsch global alignment behavior.

**Bug 3:** Neighbor-joining: after merging nodes i and j, the distance from new node u to remaining node k is `(d[i,k] + d[j,k] - d[i,j]) / 2`. The buggy code forgets to subtract `d[i,j]`.

</details>

In [ ]:
import numpy as np

def sw_buggy(s1, s2, match=2, mismatch=-1, gap=-2):
    n, m = len(s1), len(s2)
    dp = np.zeros((n+1, m+1), dtype=int)
    for i in range(1, n+1):
        for j in range(1, m+1):
            sc = match if s1[i-1]==s2[j-1] else mismatch
            diag = dp[i-1][j-1] + sc
            up   = dp[i-1][j] + gap
            left = dp[i][j-1] + gap
            # BUG 1: missing 0 in max — allows negative scores (NW not SW)
            dp[i][j] = max(diag, up, left)
    
    # BUG 2: starts traceback from corner, not from max
    i, j = n, m  # should be: i,j = np.unravel_index(np.argmax(dp), dp.shape)
    align1, align2 = [], []
    while i>0 and j>0 and dp[i][j] > 0:
        sc = match if s1[i-1]==s2[j-1] else mismatch
        if dp[i][j] == dp[i-1][j-1] + sc:
            align1.append(s1[i-1]); align2.append(s2[j-1]); i-=1; j-=1
        elif dp[i][j] == dp[i-1][j] + gap:
            align1.append(s1[i-1]); align2.append('-'); i-=1
        else:
            align1.append('-'); align2.append(s2[j-1]); j-=1
    return "".join(reversed(align1)), "".join(reversed(align2)), dp.max()

s1, s2 = "ATCGAT", "GCTATC"
a1, a2, score = sw_buggy(s1, s2)
print(f"SW alignment: {a1}")
print(f"             {a2}")
print(f"Score: {score} (correct SW max should be ~8 for ATCGAT vs GCTATC)")
print("Bug 1: negative DP values mean local alignment is actually global")
print("Bug 2: traceback from corner gives wrong local alignment region")


## Notebook Complete

**You can now:**
- Solve combined alignment and phylogenetics problems at HackerRank difficulty
- Apply UPGMA, Neighbour Joining, and parsimony methods programmatically

**Where this fits:**
- Previous: 04_statistics_ml_practical
- Next: 06_mock_assessment — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes